In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
import os
from openai import OpenAI

openai_client = OpenAI(
    api_key=os.getenv("GEMINI_API_KEY"),
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

In [3]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

In [4]:
documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

### Q1: How many lesson pages
How many lesson pages are in the dataset?

In [5]:
print(len(files))
print(len(documents))

72
72


A1: 72 pages.

### Q2: Indexing and Searching
Index the documents with minsearch - make content a text field and filename a keyword field. Then search with this query:

How does the agentic loop keep calling the model until it stops?

What's the filename of the first result?

In [6]:
from minsearch import Index

index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

index.fit(documents)

In [7]:
question ="How does the agentic loop keep calling the model until it stops?"
index.search(
        question,
        num_results=1
    )

[{'content': '# The Agentic Loop\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=ePlQUcTPPjw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous lesson, we did function calling by hand. We sent a\nmessage and got back a function call. We ran it, sent the result back,\nand got the answer.\n\nThat works for one function call. It breaks down when the model wants\nto search several times, or when the first search misses the answer.\nWe don\'t know in advance how many calls the model will want. So we\nneed a loop that keeps calling the model and running tools until it\'s\ndone. An agent is exactly that.\n\n## Anatomy of an agent\n\nWith the LLM in the driver\'s seat, we have an agent. It\'s an AI\nassistant whose goal is to help the user.\n\nAn agent has three parts:\n\n- Instructions, the role and behavior we want. We pass this as the\n  `developer` message. The better the instructions, the better the\n  agent helps.\n- Tools, the functions the agent can call to carry

A2: 01-agentic-rag/lessons/14-agentic-loop.md

### Q3: RAG
Now we will build a RAG assistant on top of this data. Let's use the rag helper script we prepared during the lessons:

wget https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/rag_helper.py
RAGBase was written for the FAQ schema (section/question/answer), while our documents have filename and content.

Two solutions are possible:

Implement the RAG flow yourself
Take RAGBase and change the parts related to the FAQ schema - search (to use our index) and build_context
Build a RAG over the index from Q2 and answer the query:

How does the agentic loop keep calling the model until it stops?

Use gpt-5.4-mini. How many input (prompt) tokens did we send to the model for this request?

#### Note: I'm using gemini-2.5-flash instead.

In [8]:
!wget https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/rag_helper.py

--2026-06-22 13:23:58--  https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/rag_helper.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.111.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2134 (2.1K) [text/plain]
Saving to: ‘rag_helper.py.11’

rag_helper.py.11    100%[===================>]   2.08K  --.-KB/s    in 0s      

2026-06-22 13:23:58 (21.6 MB/s) - ‘rag_helper.py.11’ saved [2134/2134]



In [9]:
from rag_helper import RAGBase
from openai import OpenAI

openai_client = OpenAI(
    api_key=os.getenv("GEMINI_API_KEY"),
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

assistant = RAGBase(index=index, llm_client=openai_client)

question ="How does the agentic loop keep calling the model until it stops?"

answer, usage = assistant.rag(question)

print("Answer: ", answer)
print("Usage: ", usage)

Answer:  The agentic loop keeps calling the model until it stops by checking for the presence of function calls in the model's response.

Here's how it works:

1.  **Loop Initialization:** An iteration counter (`it`) is started, and a flag `has_function_calls` is set to `False` at the beginning of each iteration.
2.  **Model Call:** The model is called with the current message history, including the user's question, developer instructions, and any previous tool outputs.
3.  **Process Response:** The model's response is appended to the message history. The response is then iterated through:
    *   If an item in the response is a `function_call`, the `make_call` helper is used to execute the specified tool (e.g., `search`). The result of this tool call is then appended to the message history, and the `has_function_calls` flag is set to `True`.
    *   If an item is a `message`, it means the model is providing a conversational response.
4.  **Exit Condition:** After processing all items 

A3: 2678 prompt tokens used.

### Q4: Chunking
The lesson pages are long - some are thousands of characters. Long documents make retrieval less precise: a match deep inside a page still pulls in the whole page. A common fix is chunking: split each page into smaller, overlapping pieces and index those instead.

gitsource has a helper for this: chunk_documents. It uses a sliding window - a window of size characters slides across the text in steps of step characters, and each window position becomes one chunk.

- Each chunk is a window of size characters of the page.
- The window moves forward by step characters between chunks. Since step is smaller than size, consecutive chunks overlap by size - step (1000) characters, so a passage split across a boundary still appears whole in one of the chunks.
- Every chunk keeps the original fields (filename) and adds start (the offset in the page) and content (the chunk text).

How many chunks do you get?

In [10]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

In [11]:
print(len(chunks))

295


A4: 295 chunks.

### Q5: RAG with chunking
Chunking makes each request smaller, because we send a smaller context to the LLM. Let's measure that.

Index the chunks from Q4 (same as before: content as a text field, filename as a keyword field), point your RAG at the chunk index, and answer the same query again - reading the input tokens the same way as in Q3.

Compare the input tokens with Q3. How many fewer input tokens does the chunked version send?

In [12]:
chunk_index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

chunk_index.fit(chunks)

In [13]:
from rag_helper import RAGBase
from openai import OpenAI

openai_client = OpenAI(
    api_key=os.getenv("GEMINI_API_KEY"),
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

chunk_assistant = RAGBase(index=chunk_index, llm_client=openai_client)

question ="How does the agentic loop keep calling the model until it stops?"

chunk_answer, chunk_usage = chunk_assistant.rag(question)

print("Answer: ", chunk_answer)
print("Usage: ", chunk_usage)

Answer:  The agentic loop keeps calling the model until it stops using a `while True` loop and an explicit break condition.

Here's how it works:
1.  **`while True` loop:** The core of the agent is a `while True` loop, which means it will continue to execute indefinitely unless a `break` statement is encountered.
2.  **Model Call:** Inside each iteration of this loop, the `openai_client.responses.create()` function is called, which invokes the model.
3.  **Function Call Detection:** The code then iterates through the model's `response.output`. It checks if any item in the output is of `item.type == "function_call"`.
4.  **`has_function_calls` flag:** A boolean variable `has_function_calls` is set to `True` if any function calls are detected in the current response.
5.  **Break Condition:** At the end of each loop iteration, the condition `if has_function_calls == False:` is checked. If no function calls were made by the model in that specific turn, it means the model has returned a fin

In [14]:
original_tokens = 2678
chunk_tokens = 621
ratio = original_tokens / chunk_tokens
print(ratio)

4.312399355877616


In [15]:
print(ratio-3, 10-ratio)
print(ratio/3, 10/ratio)

1.3123993558776164 5.687600644122384
1.4374664519592055 2.3188946975354745


A5: It is 4.3x fewer input tokens, which is nearer to 3x fewer tokens than 10x fewer tokens.

### Q6: Turning it into an agent
So far search runs once, with the exact query. Let's make it agentic: give the LLM a search tool and let it decide when (and what) to search. We suggest toyaikit, the small agent library from the module, but you can use anything you like - the OpenAI Agents SDK, PydanticAI, LangChain, or a hand-written loop.

Create a search function that uses the chunk index. Give it a type hint and a docstring - most frameworks read them to build the tool schema for you.

Build an agent with your search tool and run it (with toyaikit, the same way as in the ToyAIKit lesson). Use these instructions for the agent (they nudge it to search a few times):

You're a course teaching assistant. Answer the student's question using the search tool. Make multiple searches with different keywords before answering.

Ask it:

How does the agentic loop work, and how is it different from plain RAG?

The agent decides on its own when to search and when to answer. Count how many times it called the search tool.

How many times did the agent call search?

In [16]:
def search(query):
    return chunk_index.search(
        query,
        num_results=5
    )

In [17]:
search_tool = {
    "type": "function",
    "function": {
        "name": "search",
        "description": "Search the course lesson content for relevant information",
        "parameters": {
            "type": "object",
            "properties": {
                "query": {"type": "string", "description": "search keywords"}
            },
            "required": ["query"]
        }
    }
}

In [18]:
import json

def agent_loop(instructions, question, model="gemini-2.5-flash"):
    messages = [
        {"role": "system", "content": instructions},
        {"role": "user", "content": question}
    ]

    search_call_count = 0
    last_answer = None

    while True:
        response = openai_client.chat.completions.create(
            model=model,
            messages=messages,
            tools=[search_tool]
        )

        msg = response.choices[0].message
        assistant_msg = {"role": "assistant", "content": msg.content or ""}
        if msg.tool_calls:
            assistant_msg["tool_calls"] = [
                {
                    "id": call.id,
                    "type": "function",
                    "function": {
                        "name": call.function.name,
                        "arguments": call.function.arguments
                    }
                }
                for call in msg.tool_calls
            ]
        messages.append(assistant_msg)

        tool_calls = msg.tool_calls

        if not tool_calls:
            last_answer = msg.content
            break

        for call in tool_calls:
            args = json.loads(call.function.arguments)
            result = search(**args)
            search_call_count += 1

            messages.append({
                "role": "tool",
                "tool_call_id": call.id,
                "content": json.dumps(result)
            })

    return last_answer, search_call_count

In [19]:
instructions = """
    You're a course teaching assistant. Answer the student's question using the search tool. Make multiple searches with different keywords before answering.
""".strip()

question = "How does the agentic loop work, and how is it different from plain RAG?"
answer, count = agent_loop(instructions, question)

print("Search calls: ", count)
print("Answer: ", answer)

Search calls:  1
Answer:  The agentic loop is a dynamic process where an AI agent (powered by a Large Language Model) continuously calls the LLM, executes any tool calls it returns, sends the results back to the LLM, and repeats these steps until the model produces a final answer without further tool calls. This pattern forms the foundation of many agent frameworks, which manage the loop, conversation history, and tool execution for you.

In contrast, a plain RAG (Retrieval Augmented Generation) pipeline typically follows a fixed, three-step sequence:
1.  **Search:** Relevant information is retrieved based on the user's query (e.g., using keyword or vector search).
2.  **Build Prompt:** The retrieved search results are incorporated into a prompt along with the user's question.
3.  **LLM Call:** The LLM generates an answer based on this constructed prompt.

The key differences are:

*   **Dynamic vs. Fixed Sequence:** The agentic loop is dynamic; the agent decides what actions to take (

A6: Agent only called search for 1 time.